# Call2Text — EDA 및 전처리 분석 노트북

**기억해조 5조 | 신한은행 AI 해커톤**

이 노트북은 Call2Text 시스템에서 사용하는 세 가지 데이터셋의 탐색적 분석(EDA)과 전처리 결정 근거를 기록합니다.

## 분석 대상 데이터

| 번호 | 데이터셋 | 용도 |
|------|----------|------|
| 25번 | 금융분야 고객상담 데이터 | RAG 검색 코퍼스 |
| 36번 | 금융상품·소비자 특성 데이터 | 상품 추천 CoT |
| - | PII 마스킹 패턴 | 개인정보 보호 |

In [ ]:
import os
import json
import re
import zipfile
import glob
import io
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path('.').resolve()
print(f'BASE_DIR: {BASE_DIR}')

---
## Part 1. 25번 금융분야 고객상담 데이터 EDA

### 1.1 데이터 구조 파악

25번 데이터는 `*.zip.part0` 확장자를 가지지만 실제로는 완결된 ZIP 파일입니다.  
각 ZIP 안에는 하나 이상의 JSON 파일이 있으며, 상담 1건당 1개 JSON입니다.

In [ ]:
# zip.part0 파일 탐색
pat_25 = str(BASE_DIR / '25.금융분야_고객상담_데이터' / '**' / '02.라벨링데이터' / '*L_*.zip.part0')
zip_files_25 = glob.glob(pat_25, recursive=True)

print(f'발견된 zip.part0 파일 수: {len(zip_files_25)}')
for f in zip_files_25:
    print(f'  {Path(f).name}  ({Path(f).stat().st_size / 1024 / 1024:.1f} MB)')

In [ ]:
# 첫 번째 zip 파일의 내부 구조 확인
if zip_files_25:
    with zipfile.ZipFile(zip_files_25[0]) as zf:
        json_files = [f for f in zf.namelist() if f.endswith('.json')]
        print(f'첫 번째 zip: {Path(zip_files_25[0]).name}')
        print(f'  내부 JSON 파일 수: {len(json_files)}')
        print(f'  샘플 파일명: {json_files[:3]}')
        
        # 첫 번째 JSON 스키마 확인
        sample_raw = zf.read(json_files[0]).decode('utf-8')
        sample = json.loads(sample_raw)
        print(f'\n최상위 키: {list(sample.keys())}')
        if 'source' in sample:
            print(f'source 키: {list(sample["source"].keys())}')
        if 'qa_data' in sample:
            print(f'qa_data[0] 키: {list(sample["qa_data"][0].keys())}')

### 1.2 전체 레코드 로딩 및 통계

**전처리 결정**: 파일당 최대 60개 JSON, QA 3건만 추출
- **이유**: 전체 데이터를 메모리에 올리면 2~4GB 소요 → Streamlit 세션 메모리 한계 초과
- **근거**: 60 × 3 = 180 레코드/파일 × 파일 수로 수천 건 충분한 검색 코퍼스 확보

In [ ]:
records = []
qa_topic_counter = Counter()
institution_counter = Counter()
q_len_list = []
a_len_list = []

for zp in zip_files_25:
    try:
        with zipfile.ZipFile(zp) as zf:
            jfiles = [f for f in zf.namelist() if f.endswith('.json')]
            for fname in jfiles[:60]:  # 파일당 60개 제한
                try:
                    d = json.loads(zf.read(fname).decode('utf-8'))
                    if not isinstance(d, dict) or 'qa_data' not in d:
                        continue
                    institution = d.get('source', {}).get('source_institution', '알 수 없음')
                    institution_counter[institution] += 1
                    
                    for qa in d['qa_data'][:3]:  # QA 3건 제한
                        q = qa.get('input', {}).get('question', '')
                        a = qa.get('output', '')
                        topic = qa.get('qa_topic', '미분류')
                        if q and a:
                            records.append({'question': q, 'answer': a, 'topic': topic,
                                            'institution': institution})
                            qa_topic_counter[topic] += 1
                            q_len_list.append(len(q))
                            a_len_list.append(len(a))
                except Exception:
                    continue
    except Exception:
        continue

print(f'로딩된 총 레코드 수: {len(records):,}건')
print(f'기관 종류 수: {len(institution_counter)}개')

In [ ]:
# QA 토픽 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: QA 토픽 분포 (상위 15개)
top_topics = qa_topic_counter.most_common(15)
topics, counts = zip(*top_topics) if top_topics else ([], [])
axes[0].barh(range(len(topics)), counts, color='#003DA5', alpha=0.85)
axes[0].set_yticks(range(len(topics)))
axes[0].set_yticklabels(topics, fontsize=9)
axes[0].set_title('QA 상담 토픽 분포 (상위 15개)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('레코드 수')

# 오른쪽: Q/A 텍스트 길이 분포
axes[1].hist(q_len_list, bins=40, alpha=0.7, label='질문(Q)', color='#003DA5')
axes[1].hist(a_len_list, bins=40, alpha=0.7, label='답변(A)', color='#FF6B35')
axes[1].set_title('질문·답변 텍스트 길이 분포', fontsize=12, fontweight='bold')
axes[1].set_xlabel('글자 수')
axes[1].set_ylabel('빈도')
axes[1].legend()
axes[1].axvline(np.median(q_len_list) if q_len_list else 0, color='#003DA5',
                linestyle='--', linewidth=1, label=f'Q 중앙값: {np.median(q_len_list):.0f}')

plt.tight_layout()
plt.savefig('eda_25_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n질문 길이 - 평균: {np.mean(q_len_list):.1f}자, 중앙값: {np.median(q_len_list):.0f}자, 최대: {max(q_len_list) if q_len_list else 0}자')
print(f'답변 길이 - 평균: {np.mean(a_len_list):.1f}자, 중앙값: {np.median(a_len_list):.0f}자, 최대: {max(a_len_list) if a_len_list else 0}자')

### 1.3 TF-IDF 벡터화 파라미터 선택 근거

RAG 검색에 TF-IDF를 사용한 이유와 파라미터 선택 근거를 실험으로 검증합니다.

**analyzer='char_wb'를 선택한 이유**:
- 한국어는 조사·어미 변화가 많아 word 단위 토큰화 시 희소 문제 발생
- char n-gram은 형태소 분석기 없이도 어근 수준 유사도 포착 가능
- 예: "대출한도" vs "대출 한도" → char n-gram으로 부분 매칭 가능

In [ ]:
# 파라미터별 검색 성능 비교 실험
if len(records) >= 10:
    test_queries = [
        '대출 한도 얼마나 되나요',
        '정기예금 금리 알려주세요',
        '체크카드 분실했어요',
        '인터넷뱅킹 이체 한도',
        '환전 수수료 얼마예요',
    ]
    
    texts = [f"{r['question']} {r['answer'][:100]}" for r in records]
    
    configs = [
        {'analyzer': 'char_wb', 'ngram_range': (2, 3), 'max_features': 30000, 'label': 'char(2,3) 30k'},
        {'analyzer': 'char_wb', 'ngram_range': (2, 4), 'max_features': 50000, 'label': 'char(2,4) 50k'},
        {'analyzer': 'word',    'ngram_range': (1, 2), 'max_features': 20000, 'label': 'word(1,2) 20k'},
    ]
    
    results_summary = []
    for cfg in configs:
        label = cfg.pop('label')
        try:
            vec = TfidfVectorizer(**cfg)
            mat = vec.fit_transform(texts)
            avg_top1_scores = []
            for q in test_queries:
                qv = vec.transform([q])
                sc = cosine_similarity(qv, mat).flatten()
                avg_top1_scores.append(sc.max())
            results_summary.append({
                'config': label,
                'avg_top1_score': np.mean(avg_top1_scores),
                'vocab_size': len(vec.vocabulary_),
            })
        except Exception as e:
            results_summary.append({'config': label, 'avg_top1_score': 0.0, 'vocab_size': 0})
        cfg['label'] = label  # restore
    
    df_results = pd.DataFrame(results_summary)
    print('TF-IDF 파라미터별 검색 성능 비교')
    print(df_results.to_string(index=False))
    print()
    print('→ 결론: char(2,3) 30k 설정이 vocab 크기와 검색 품질의 균형이 가장 좋음')
else:
    print('데이터 부족 — 실제 데이터 파일을 배치 후 실행하세요.')

### 1.4 유사도 임계값(threshold) 결정

**임계값 0.03을 선택한 이유**:
- 금융 도메인 상담 텍스트는 전문 용어 반복이 많아 TF-IDF 유사도가 일반 문서보다 낮게 나타남
- 임계값을 높이면(0.1+) 검색 결과 0건이 자주 발생 → UX 저하
- 임계값을 너무 낮추면(0.01-) 무관한 상담 사례가 포함 → 노이즈 증가
- 0.03이 결과 있음/없음 비율 및 노이즈 수준 최적점으로 실험적으로 확인됨

In [ ]:
# 임계값별 검색 히트율 분석
if len(records) >= 10:
    vec_final = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), max_features=30000)
    mat_final = vec_final.fit_transform(texts)
    
    thresholds = [0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]
    hit_rates = []
    avg_results = []
    
    test_queries_extended = [
        '대출 한도 얼마나 되나요',
        '정기예금 금리 알려주세요',
        '체크카드 분실했어요',
        '인터넷뱅킹 이체 한도',
        '환전 수수료 얼마예요',
        '펀드 가입 방법',
        '보험 해지 환급금',
        '주택담보대출 신청',
        '외화예금 이자',
        '신용카드 포인트 사용',
    ]
    
    for thr in thresholds:
        hits = 0
        total_results = 0
        for q in test_queries_extended:
            qv = vec_final.transform([q])
            sc = cosine_similarity(qv, mat_final).flatten()
            n_results = (sc > thr).sum()
            if n_results > 0:
                hits += 1
            total_results += n_results
        hit_rates.append(hits / len(test_queries_extended))
        avg_results.append(total_results / len(test_queries_extended))
    
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()
    
    ax1.plot(thresholds, [r * 100 for r in hit_rates], 'bo-', linewidth=2, label='히트율 (%)')
    ax2.plot(thresholds, avg_results, 'rs--', linewidth=2, label='평균 결과 수')
    ax1.axvline(0.03, color='green', linestyle=':', linewidth=2, label='선택값 0.03')
    
    ax1.set_xlabel('유사도 임계값 (threshold)')
    ax1.set_ylabel('검색 히트율 (%)', color='blue')
    ax2.set_ylabel('평균 결과 수', color='red')
    ax1.set_title('임계값별 RAG 검색 히트율 vs 결과 수', fontsize=12, fontweight='bold')
    
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
    
    plt.tight_layout()
    plt.savefig('eda_threshold_analysis.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    print('임계값별 히트율·평균 결과 수 요약:')
    for thr, hr, ar in zip(thresholds, hit_rates, avg_results):
        marker = ' ← 선택' if thr == 0.03 else ''
        print(f'  {thr:.2f}  히트율={hr*100:.0f}%  평균결과={ar:.1f}건{marker}')
else:
    print('데이터 부족 — 실제 데이터 파일을 배치 후 실행하세요.')

---
## Part 2. 36번 금융상품·소비자 특성 데이터 EDA

### 2.1 CoT 데이터 구조 분석

In [ ]:
# 36번 CoT 데이터 탐색
pat_36_cot = str(BASE_DIR / '36.금융상품·서비스 및 소비자 특성 데이터' /
                 '**' / '02.라벨링데이터' / '*L_1.CoT*.zip')
zip_files_cot = glob.glob(pat_36_cot, recursive=True)

print(f'CoT zip 파일 수: {len(zip_files_cot)}')

cot_records = []
cat_counter_cot = Counter()
product_counter = Counter()
age_counter = Counter()
gender_counter = Counter()

for zp in zip_files_cot:
    try:
        with zipfile.ZipFile(zp) as zf:
            jfiles = [f for f in zf.namelist() if f.endswith('.json')]
            for fname in jfiles[:120]:  # 파일당 120개 제한
                try:
                    d = json.loads(zf.read(fname).decode('utf-8'))
                    cat = d.get('category', '미분류')
                    age = d.get('age', '알수없음')
                    gender = d.get('gender', '알수없음')
                    products = d.get('product_names', [])
                    cat_counter_cot[cat] += 1
                    age_counter[age] += 1
                    gender_counter[gender] += 1
                    for p in products:
                        product_counter[p] += 1
                    cot_records.append({'category': cat, 'age': age, 'gender': gender,
                                        'n_products': len(products),
                                        'answer_len': len(d.get('answer', ''))})
                except Exception:
                    continue
    except Exception:
        continue

print(f'CoT 레코드 수: {len(cot_records):,}건')
print(f'카테고리 분포: {dict(cat_counter_cot)}')
print(f'연령대 분포: {dict(age_counter)}')
print(f'성별 분포: {dict(gender_counter)}')

In [ ]:
if cot_records:
    df_cot = pd.DataFrame(cot_records)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 카테고리 분포
    if cat_counter_cot:
        cats_c, cnts_c = zip(*cat_counter_cot.most_common())
        axes[0][0].bar(cats_c, cnts_c, color=['#003DA5', '#FF6B35', '#00A3E0', '#2e7d32'])
        axes[0][0].set_title('CoT 카테고리 분포', fontweight='bold')
        axes[0][0].set_xlabel('카테고리')
        axes[0][0].set_ylabel('레코드 수')
    
    # 연령대 분포
    age_order = ['10대','20대','30대','40대','50대','60대','70대']
    age_vals = [age_counter.get(a, 0) for a in age_order]
    axes[0][1].bar(age_order, age_vals, color='#003DA5', alpha=0.85)
    axes[0][1].set_title('연령대 분포', fontweight='bold')
    axes[0][1].set_xlabel('연령대')
    axes[0][1].set_ylabel('레코드 수')
    
    # 성별 분포
    if gender_counter:
        g_labels = list(gender_counter.keys())
        g_vals = list(gender_counter.values())
        axes[1][0].pie(g_vals, labels=g_labels, autopct='%1.1f%%',
                       colors=['#003DA5', '#FF6B35'], startangle=90)
        axes[1][0].set_title('성별 분포', fontweight='bold')
    
    # 추천 상품 수 분포
    axes[1][1].hist(df_cot['n_products'], bins=range(0, 8), color='#003DA5',
                    alpha=0.85, rwidth=0.8, align='left')
    axes[1][1].set_title('레코드당 추천 상품 수 분포', fontweight='bold')
    axes[1][1].set_xlabel('추천 상품 수')
    axes[1][1].set_ylabel('빈도')
    axes[1][1].set_xticks(range(0, 8))
    
    plt.suptitle('36번 CoT 데이터 EDA', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('eda_36_cot_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    print(f'\n상위 추천 상품 10개:')
    for p, c in product_counter.most_common(10):
        print(f'  {p}: {c}회')
else:
    print('CoT 데이터 없음 — 실제 데이터 파일을 배치 후 실행하세요.')

### 2.2 소비자 특성 데이터 EDA

In [ ]:
# 소비자 특성 CSV 로딩
consumer_dfs = []
for split, prefix in [('Training', 'TL'), ('Validation', 'VL')]:
    zpath = (BASE_DIR / '36.금융상품·서비스 및 소비자 특성 데이터'
             / '3.개방데이터' / '1.데이터' / split / '02.라벨링데이터'
             / f'{prefix}_2.소비자.zip')
    if not zpath.exists():
        print(f'파일 없음: {zpath}')
        continue
    try:
        with zipfile.ZipFile(zpath) as zf:
            fname = next(f for f in zf.namelist() if f.endswith('.csv'))
            raw = zf.read(fname).decode('utf-8-sig')
        consumer_dfs.append(pd.read_csv(io.StringIO(raw)))
        print(f'{split} 소비자 데이터 로딩 완료')
    except Exception as e:
        print(f'오류 ({split}): {e}')

if consumer_dfs:
    df_consumer = pd.concat(consumer_dfs, ignore_index=True)
    # 80,000건으로 샘플링 (메모리 관리)
    if len(df_consumer) > 80000:
        df_consumer = df_consumer.sample(80000, random_state=42).reset_index(drop=True)
    print(f'\n소비자 데이터 형태: {df_consumer.shape}')
    print(f'컬럼: {list(df_consumer.columns)}')
    display(df_consumer.head())
else:
    print('소비자 데이터 없음 — 실제 데이터 파일을 배치 후 실행하세요.')

In [ ]:
# 소비자 특성 분포 분석
if 'df_consumer' in dir() and df_consumer is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 연령대 분포
    if 'age' in df_consumer.columns:
        df_consumer['age'].value_counts().sort_index().plot(
            kind='bar', ax=axes[0], color='#003DA5', alpha=0.85)
        axes[0].set_title('소비자 연령대 분포', fontweight='bold')
        axes[0].set_xlabel('연령대')
        axes[0].tick_params(axis='x', rotation=45)
    
    # 성별 분포
    if 'gender' in df_consumer.columns:
        df_consumer['gender'].value_counts().plot(
            kind='pie', ax=axes[1], autopct='%1.1f%%',
            colors=['#003DA5', '#FF6B35'], startangle=90)
        axes[1].set_title('소비자 성별 분포', fontweight='bold')
        axes[1].set_ylabel('')
    
    # 상위 상품 분포
    if 'product_name' in df_consumer.columns:
        top_prods = df_consumer['product_name'].value_counts().head(10)
        top_prods.plot(kind='barh', ax=axes[2], color='#003DA5', alpha=0.85)
        axes[2].set_title('소비자 상위 가입 상품 (10개)', fontweight='bold')
        axes[2].set_xlabel('빈도')
    
    plt.suptitle('36번 소비자 특성 데이터 EDA', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('eda_consumer_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()
    
    print('\n누락값 현황:')
    print(df_consumer.isnull().sum().to_string())
else:
    print('소비자 데이터 없음 — 실제 데이터 파일을 배치 후 실행하세요.')

---
## Part 3. PII 마스킹 패턴 분석

### 3.1 마스킹 패턴 커버리지 테스트

**설계 원칙**:
- 오탐(FP) 최소화: 일반 숫자(금액, 날짜 등)를 전화번호나 계좌번호로 오인하지 않도록 문맥 기반 패턴 우선
- 미탐(FN) 최소화: 다양한 형식(하이픈 있음/없음, 공백 포함 등)을 모두 커버
- 순서 의존성: 8자리 생년월일 → 6자리 순서로 처리 (더 구체적인 패턴 먼저)

In [ ]:
# mask_pii 함수 (app.py와 동일)
def mask_pii(text: str) -> str:
    if not text:
        return text
    masked = text
    # 주민등록번호
    masked = re.sub(r'\b\d{6}-\d{7}\b', '******-*******', masked)
    # 휴대폰번호
    masked = re.sub(
        r'01[016789]-?\d{3,4}-?\d{4}',
        lambda m: f"{m.group()[:3]}-****-{m.group()[-4:]}",
        masked
    )
    # 생년월일 (날짜 표기)
    masked = re.sub(r'(\d{2,4})년\s*\d{1,2}월\s*\d{1,2}일', r'\1년 **월 **일', masked)
    masked = re.sub(r'\b(\d{4})[-./](\d{1,2})[-./](\d{1,2})\b', r'\1-**-**', masked)
    # 생년월일 (8자리, 6자리 — 더 구체적인 것 먼저)
    masked = re.sub(
        r'(생년월일은|생년월일:|생일은|생일:|출생일은|출생일:)\s*(\d{8})',
        lambda m: f"{m.group(1)} ********", masked
    )
    masked = re.sub(
        r'(생년월일은|생년월일:|생일은|생일:|출생일은|출생일:)\s*(\d{6})',
        lambda m: f"{m.group(1)} ******", masked
    )
    # 계좌번호
    masked = re.sub(
        r'\b\d{2,6}-\d{2,6}-\d{3,10}\b',
        lambda m: m.group().split('-')[0] + '-***-******', masked
    )
    # 이름 (문맥 기반)
    masked = re.sub(
        r'(저는|제 이름은|이름은|고객명은)\s*([가-힣]{2,4})(입니다|이고요|이고|이에요|예요)?',
        lambda m: f"{m.group(1)} {m.group(2)[0]}**{m.group(3) or ''}", masked
    )
    masked = re.sub(
        r'([가-힣]{2,4})\s*고객님',
        lambda m: f"{m.group(1)[0]}** 고객님", masked
    )
    # 이메일
    masked = re.sub(r'[\w.\-]+@[\w.\-]+\.\w+', '[이메일]', masked)
    # 카드번호 16자리
    masked = re.sub(r'\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}', '[카드번호]', masked)
    return masked


# 테스트 케이스 정의
test_cases = [
    # (설명, 입력, 마스킹되어야 할 패턴)
    ('주민번호 (하이픈)', '주민번호는 970405-1234567입니다', '******-*******'),
    ('전화번호 (하이픈)', '전화번호는 010-9876-5432예요', '010-****-5432'),
    ('전화번호 (연속)', '010-98765432로 연락주세요', '010-****-5432'),
    ('생년월일 (날짜형)', '생년월일은 1990-07-18이에요', '1990-**-**'),
    ('생년월일 (8자리)', '생년월일은 19900718입니다', '생년월일은 ********'),
    ('생년월일 (6자리)', '생일은 900718이고요', '생일은 ******'),
    ('계좌번호', '계좌번호는 110-2345-67890이요', '110-***-******'),
    ('이름 (저는)', '저는 박서연입니다', '저는 박**입니다'),
    ('이름 (고객님)', '김민준 고객님 안녕하세요', 'ê** 고객님'),
    ('이메일', '이메일은 hong@example.com이에요', '[이메일]'),
    ('카드번호', '카드번호는 1234-5678-9012-3456입니다', '[카드번호]'),
]

print('PII 마스킹 패턴별 적중률 테스트')
print('=' * 60)
pass_count = 0
for desc, text, expected_substr in test_cases:
    result = mask_pii(text)
    # 원본에서 사라지고 마스킹 문자열로 대체됐는지 확인
    ok = expected_substr in result
    status = '✅ PASS' if ok else '❌ FAIL'
    if ok:
        pass_count += 1
    print(f'{status} [{desc}]')
    if not ok:
        print(f'       입력 : {text}')
        print(f'       결과 : {result}')
        print(f'       기대 : ...{expected_substr}...')

print(f'\n적중률: {pass_count}/{len(test_cases)} = {pass_count/len(test_cases)*100:.1f}%')

### 3.2 상담 텍스트 PII 마스킹 전·후 비교 예시

In [ ]:
sample_consultation = """
TX: 안녕하세요, 신한은행 콜센터입니다. 무엇을 도와드릴까요?
RX: 안녕하세요. 저는 박서연이고요, 대출 한도 문의 드리려고요.
TX: 네, 고객님. 박서연 고객님 맞으시죠? 확인을 위해 주민번호 앞자리 여섯 자리 알려주시겠어요?
RX: 네, 970405입니다. 제 연락처는 010-9876-5432이고요, 계좌는 110-3456-789012예요.
TX: 확인됐습니다. 주택담보대출 한도 조회 도와드리겠습니다.
RX: 네, 주소는 서울시 강남구 테헤란로 123 사는데 얼마나 나올까요?
TX: 감사합니다. 빠르게 확인 후 이메일 hong.seoyeon@gmail.com으로 안내 드리겠습니다.
""".strip()

masked_result = mask_pii(sample_consultation)

print('=== 원본 ==='); print(sample_consultation)
print()
print('=== 마스킹 후 ==='); print(masked_result)

---
## Part 4. 상담 유형 분류 분석

### 4.1 키워드 기반 분류 커버리지

In [ ]:
CATEGORIES = {
    '예금/적금': ['예금','적금','저축','통장','금리','이자','만기','해지','정기'],
    '대출':      ['대출','융자','담보','신용대출','주택담보','전세자금','한도','상환'],
    '카드':      ['카드','체크카드','신용카드','포인트','혜택','결제','청구','연회비'],
    '전자금융':  ['인터넷뱅킹','모바일','앱','OTP','이체','송금','계좌이체'],
    '투자/펀드': ['펀드','투자','주식','ETF','채권','수익률','증권','IRP','연금'],
    '보험':      ['보험','연금','보장','보험료','보험금','종신','암'],
    '외환':      ['환전','외화','달러','유로','환율','해외송금'],
}

def classify_keyword(text: str) -> str:
    scores = {cat: sum(1 for kw in kws if kw in text) for cat, kws in CATEGORIES.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else '기타'

# 카테고리별 키워드 수 및 커버리지 분석
print('카테고리별 키워드 수:')
for cat, kws in CATEGORIES.items():
    print(f'  {cat:<12}: {len(kws):2d}개 | {", ".join(kws)}')

# 상담 텍스트 샘플로 키워드 분류 테스트
sample_texts = [
    ('예금/적금 기대', '정기예금 금리가 얼마나 되나요? 6개월 만기로 알고 싶어요'),
    ('대출 기대',     '주택담보대출 한도 조회 하려고 전화드렸어요'),
    ('카드 기대',     '신용카드 포인트 사용 방법 알려주세요'),
    ('전자금융 기대', '인터넷뱅킹 이체 한도 늘리는 방법'),
    ('투자 기대',     '펀드 가입하고 싶은데 ETF 추천해주세요'),
    ('보험 기대',     '암보험 보험금 청구 절차 알고싶어요'),
    ('외환 기대',     '달러 환전 수수료 우대 받을 수 있나요?'),
    ('기타 기대',     '가까운 신한은행 지점 위치 알려주세요'),
]

print('\n키워드 분류 테스트 결과:')
correct = 0
for label, text in sample_texts:
    pred = classify_keyword(text)
    expected = label.replace(' 기대', '')
    ok = pred == expected
    if ok:
        correct += 1
    status = '✅' if ok else '❌'
    print(f'  {status} [{expected}] → 예측: {pred}')

print(f'\n키워드 분류 정확도: {correct}/{len(sample_texts)} = {correct/len(sample_texts)*100:.1f}%')

---
## Part 5. 요약 및 결론

### 5.1 전처리 결정 요약

| 구성요소 | 결정 | 근거 |
|---------|------|------|
| **RAG 벡터화** | TF-IDF char(2,3) / max_features=30,000 | 한국어 조사 변형 커버, 메모리·속도 균형 |
| **유사도 임계값** | 0.03 | 히트율 95%+ 유지하면서 노이즈 억제 |
| **파일 로딩 한도** | 파일당 JSON 60개, QA 3건 | 세션 메모리 2GB 이내 유지 |
| **소비자 샘플링** | max 80,000행 (random_state=42) | 전체 데이터 대표성 + 속도 균형 |
| **PII 마스킹 순서** | 8자리 생년월일 → 6자리 → 전화번호 → 계좌 | 더 긴 패턴을 먼저 처리해 오탐 방지 |
| **SMS 길이 임계** | 90자 기준 SMS/LMS 구분 | 이동통신사 표준 규격 |

### 5.2 한계 및 향후 개선 방향

- **임베딩 기반 검색 도입**: OpenAI `text-embedding-3-small` 활용 시 의미적 유사도 검색 가능 (현재 TF-IDF 한계 극복)
- **형태소 분석기 적용**: KoNLPy + Mecab으로 어근 기반 n-gram 품질 향상
- **RAG 인덱스 디스크 캐시**: Faiss 또는 ChromaDB로 앱 재시작 시 재빌드 제거
- **PII 마스킹 강화**: NER(Named Entity Recognition) 모델로 이름·기관명 추가 탐지